# Notebook 04 — paper-facing figures and tables

This notebook reads frozen outputs from Notebooks 01–03 and creates the paper-facing source tables and plots. It does not perform cell QC, clustering, donor selection, gene filtering, or differential-expression fitting.

The old working notebook contained very long figure-assembly cells and an obsolete GSE138852 branch. This public version uses the final donor reconstruction and the completed three-accession external synthesis. Supplementary Figure S4 is exported as separate panels rather than repeatedly reassembling raster images in Python.

## Setup

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2, norm

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
except ImportError:
    pass

ROOT = Path("/content/drive/MyDrive/AD_Astrocyte_Paper_01")
FIG_MAIN = ROOT / "manuscript_figures" / "main"
FIG_SUPP = ROOT / "manuscript_figures" / "supplementary"
TAB_SUPP = ROOT / "manuscript_tables" / "supplementary"

for p in (FIG_MAIN, FIG_SUPP, TAB_SUPP):
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["svg.fonttype"] = "none"

In [ ]:
def require(*paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        raise FileNotFoundError("Missing frozen input(s):\n" + "\n".join(missing))

def save_figure(fig, stem, folder):
    for ext in ("png", "pdf", "svg"):
        kwargs = {"dpi": 600} if ext == "png" else {}
        fig.savefig(folder / f"{stem}.{ext}", bbox_inches="tight", **kwargs)

def creb5_row(path):
    df = pd.read_csv(path, compression="infer")
    if "gene_symbol" in df.columns:
        hit = df.loc[df["gene_symbol"].astype(str).eq("CREB5")]
    else:
        key = "feature_id" if "feature_id" in df.columns else "gene_id"
        hit = df.loc[df[key].astype(str).eq("CREB5")]
    if len(hit) != 1:
        raise RuntimeError(f"Expected one CREB5 row in {path.name}; found {len(hit)}")
    return hit.iloc[0]

## Final CREB5 evidence table

In [ ]:
ORIGINAL = ROOT / "CREB5_three_cohort_five_analysis_effects_v1.csv"
EV = ROOT / "external_validation"

DE188 = EV / "GSE188545" / "GSE188545_astrocyte_pseudobulk_DESeq2_genomewide_v1.csv.gz"
DE138 = EV / "GSE138852" / "GSE138852_astrocyte_pseudobulk_DESeq2_genomewide_v1.csv.gz"
DE174 = EV / "GSE174367" / "GSE174367_DESeq2_primary_genomewide_v1.csv.gz"

require(ORIGINAL, DE188, DE138, DE174)

original = pd.read_csv(ORIGINAL)
original = original.loc[original["gene"].astype(str).eq("CREB5")].copy()

keep = ["cohort", "region", "log2FC", "lfcSE", "CI95_low", "CI95_high", "pvalue", "padj"]
original = original[keep].rename(columns={"cohort": "dataset", "padj": "FDR"})
assert len(original) == 5

In [ ]:
external_specs = [
    ("GSE188545", "MTG", creb5_row(DE188)),
    ("GSE138852", "EC", creb5_row(DE138)),
    ("GSE174367", "PFC", creb5_row(DE174)),
]

rows = []
for dataset, region, r in external_specs:
    rows.append({
        "dataset": dataset,
        "region": region,
        "log2FC": float(r["log2FoldChange"]),
        "lfcSE": float(r["lfcSE"]),
        "CI95_low": float(r.get("CI95_low", r["log2FoldChange"] - 1.96*r["lfcSE"])),
        "CI95_high": float(r.get("CI95_high", r["log2FoldChange"] + 1.96*r["lfcSE"])),
        "pvalue": float(r["pvalue"]),
        "FDR": float(r["padj"]),
    })

external = pd.DataFrame(rows)
effects = pd.concat([original, external], ignore_index=True)
effects["positive"] = effects["log2FC"] > 0
effects["nominal_p_lt_0_05"] = effects["pvalue"] < 0.05
effects["FDR_lt_0_05"] = effects["FDR"] < 0.05

assert len(effects) == 8
assert effects["positive"].all()
assert effects["nominal_p_lt_0_05"].sum() == 5
assert effects["FDR_lt_0_05"].sum() == 2

effects.to_csv(FIG_MAIN / "Figure1_CREB5_primary_effects_source.csv", index=False)
effects

## External-validation synthesis

In [ ]:
def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    ranked = p[order]
    adj_ranked = np.maximum.accumulate([(len(p)-i)*v for i, v in enumerate(ranked)])
    out = np.empty(len(p))
    out[order] = np.minimum(adj_ranked, 1.0)
    return out

external["Holm_p"] = holm(external["pvalue"])

y = external["log2FC"].to_numpy(float)
se = external["lfcSE"].to_numpy(float)
w = 1 / se**2

fixed = np.sum(w*y) / np.sum(w)
fixed_se = np.sqrt(1 / np.sum(w))
fixed_p = 2 * norm.sf(abs(fixed/fixed_se))
Q = np.sum(w*(y-fixed)**2)
df = len(y)-1
I2 = max(0.0, (Q-df)/Q) * 100

C = np.sum(w) - np.sum(w**2)/np.sum(w)
tau2 = max(0.0, (Q-df)/C)
wr = 1 / (se**2 + tau2)
random = np.sum(wr*y) / np.sum(wr)
random_se = np.sqrt(1 / np.sum(wr))
random_p = 2 * norm.sf(abs(random/random_se))

In [ ]:
assert np.isclose(fixed, 0.677998, atol=5e-6)
assert np.isclose(fixed_se, 0.195831, atol=5e-6)
assert np.isclose(fixed_p, 0.000535861203005, rtol=1e-6)
assert np.isclose(I2, 58.57, atol=0.02)
assert np.isclose(random, 0.670385, atol=5e-6)
assert np.isclose(random_p, 0.0316230963615, rtol=1e-6)

synthesis = pd.DataFrame([
    ["Fixed effect", fixed, fixed_se, fixed-1.96*fixed_se, fixed+1.96*fixed_se, fixed_p],
    ["DerSimonian-Laird sensitivity", random, random_se, random-1.96*random_se, random+1.96*random_se, random_p],
], columns=["analysis", "log2FC", "SE", "CI95_low", "CI95_high", "pvalue"])

external.to_csv(TAB_SUPP / "Supplementary_Table_S3_External_Validation.csv", index=False)
synthesis.to_csv(TAB_SUPP / "Supplementary_Table_S3_External_Synthesis.csv", index=False)

## Figure 1 — primary effects and external pooled estimate

In [ ]:
plot = effects.copy()
plot["label"] = plot["dataset"] + " — " + plot["region"]
plot = pd.concat([
    plot,
    pd.DataFrame([{
        "label": "External pooled — fixed effect",
        "log2FC": fixed,
        "CI95_low": fixed - 1.96*fixed_se,
        "CI95_high": fixed + 1.96*fixed_se,
        "pvalue": fixed_p,
    }])
], ignore_index=True)

fig, ax = plt.subplots(figsize=(8.0, 5.3))
ypos = np.arange(len(plot))[::-1]

for y0, r in zip(ypos, plot.itertuples()):
    ax.errorbar(
        r.log2FC, y0,
        xerr=[[r.log2FC-r.CI95_low], [r.CI95_high-r.log2FC]],
        fmt="D" if "pooled" in r.label else "o",
        capsize=3
    )

ax.axvline(0, linestyle="--", linewidth=1)
ax.set_yticks(ypos, plot["label"])
ax.set_xlabel("CREB5 log2 fold change")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_figure(fig, "Figure1_CREB5_primary_effects", FIG_MAIN)
plt.show()

## Figure 2 — GSE157827 robustness

In [ ]:
ROBUST = ROOT / "GSE157827_CREB5_robustness_summary_v1.csv"
require(ROBUST)
rob = pd.read_csv(ROBUST)

required = {"analysis", "log2FC", "lfcSE", "pvalue"}
if not required.issubset(rob.columns):
    raise RuntimeError(f"Unexpected robustness table columns: {list(rob.columns)}")

rob["CI95_low"] = rob.get("CI95_low", rob["log2FC"] - 1.96*rob["lfcSE"])
rob["CI95_high"] = rob.get("CI95_high", rob["log2FC"] + 1.96*rob["lfcSE"])
rob.to_csv(FIG_MAIN / "Figure2_GSE157827_robustness_source.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ypos = np.arange(len(rob))[::-1]

ax.errorbar(
    rob["log2FC"], ypos,
    xerr=[rob["log2FC"]-rob["CI95_low"], rob["CI95_high"]-rob["log2FC"]],
    fmt="o", capsize=3
)
ax.axvline(0, linestyle="--", linewidth=1)
ax.set_yticks(ypos, rob["analysis"].astype(str))
ax.set_xlabel("CREB5 log2 fold change (AD vs Control)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_figure(fig, "Figure2_GSE157827_CREB5_robustness", FIG_MAIN)
plt.show()

## Figure 3 — prospective external validation

In [ ]:
fig = plt.figure(figsize=(8.0, 6.0))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 3], hspace=0.25)

ax0 = fig.add_subplot(gs[0])
steps = ["Lock protocol", "Donor pseudobulk", "Freeze genome-wide DE", "Reveal CREB5", "Holm + meta-analysis"]
x = np.arange(len(steps))
ax0.plot(x, np.zeros_like(x), marker="o")
for xi, label in zip(x, steps):
    ax0.text(xi, -0.16, label, ha="center", va="top", fontsize=8)
ax0.set_xlim(-0.3, len(steps)-0.7)
ax0.set_ylim(-0.55, 0.25)
ax0.axis("off")

ax1 = fig.add_subplot(gs[1])
ypos = np.arange(len(external))[::-1]
for y0, r in zip(ypos, external.itertuples()):
    lo = r.log2FC - 1.96*r.lfcSE
    hi = r.log2FC + 1.96*r.lfcSE
    ax1.errorbar(r.log2FC, y0, xerr=[[r.log2FC-lo], [hi-r.log2FC]], fmt="o", capsize=3)

ax1.errorbar(fixed, -1, xerr=1.96*fixed_se, fmt="D", capsize=3)
ax1.errorbar(random, -2, xerr=1.96*random_se, fmt="D", mfc="none", capsize=3)
ax1.axvline(0, linestyle="--", linewidth=1)
ax1.set_yticks([2,1,0,-1,-2], list(external["dataset"]) + ["Fixed pooled", "Random sensitivity"])
ax1.set_xlabel("CREB5 log2 fold change")
ax1.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
save_figure(fig, "Figure3_external_validation", FIG_MAIN)
plt.show()

## Supplementary Figure S1 — GSE157827 QC and Scrublet audit

In [ ]:
G157 = ROOT / "GSE157827_confirmatory_validation"
QC_GROUP = G157 / "GSE157827_QC_group_reconciliation_v1.csv"
QC_DONOR = G157 / "GSE157827_donor_QC_review_v1.csv"
SCRUB = G157 / "GSE157827_scrublet_sample_summary_v1_1.csv"

if not SCRUB.exists():
    SCRUB = G157 / "GSE157827_scrublet_sample_summary_v1.csv"

require(QC_GROUP, QC_DONOR, SCRUB)
qc_group = pd.read_csv(QC_GROUP)
qc_donor = pd.read_csv(QC_DONOR)
scrub = pd.read_csv(SCRUB)

assert int(qc_group["reproduced_QC_pass"].sum()) == 169506
assert int(pd.to_numeric(scrub["retained_singlets"], errors="coerce").sum()) == 159144

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))

a = qc_group.set_index("diagnosis")[["published_final_nuclei", "reproduced_QC_pass"]]
a.plot.bar(ax=axes[0], rot=0)
axes[0].set_ylabel("Nuclei")
axes[0].set_title("Published vs reproduced QC")

axes[1].plot(
    np.arange(len(qc_donor)),
    qc_donor["author_QC_pass_percent"],
    "o"
)
axes[1].axhline(80, linestyle="--", linewidth=1)
axes[1].set_xticks(np.arange(len(qc_donor)), qc_donor["sample_label"], rotation=60, ha="right", fontsize=7)
axes[1].set_ylabel("Author-QC retention (%)")

axes[2].plot(np.arange(len(scrub)), scrub["detected_doublet_percent"], "o")
axes[2].set_xticks(np.arange(len(scrub)), scrub["sample_label"], rotation=60, ha="right", fontsize=7)
axes[2].set_ylabel("Detected doublets (%)")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
save_figure(fig, "Figure_S1_GSE157827_QC_and_Scrublet", FIG_SUPP)
plt.show()

## Supplementary Figure S2 — annotation robustness

In [ ]:
PRIMARY_COUNTS = G157 / "GSE157827_STEP9F_primary_astrocyte_counts_per_donor_v1.csv"
ALT_COUNTS = G157 / "GSE157827_STEP15C_alternative_astrocytes_per_donor_v1.csv"
ALT_COMP = G157 / "GSE157827_STEP15C_annotation_composition_v1.csv"
ALT_MARKERS = G157 / "GSE157827_STEP15C_cluster_marker_profiles_r08_v1.csv"

require(PRIMARY_COUNTS, ALT_COUNTS, ALT_COMP, ALT_MARKERS)

primary_counts = pd.read_csv(PRIMARY_COUNTS)
alt_counts = pd.read_csv(ALT_COUNTS)
alt_comp = pd.read_csv(ALT_COMP)
alt_markers = pd.read_csv(ALT_MARKERS)

In [ ]:
counts = primary_counts[["GSM", "cluster_based_astrocytes"]].merge(
    alt_counts[["GSM", "alternative_astrocyte_nuclei"]], on="GSM"
)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
axes[0].scatter(counts["cluster_based_astrocytes"], counts["alternative_astrocyte_nuclei"])
lim = max(counts.iloc[:, 1:].to_numpy().max(), 1)
axes[0].plot([0, lim], [0, lim], linestyle="--", linewidth=1)
axes[0].set_xlabel("Primary astrocytes")
axes[0].set_ylabel("Alternative astrocytes")

comp = alt_comp.sort_values("nuclei", ascending=True)
axes[1].barh(comp["assigned_lineage"], comp["nuclei"])
axes[1].set_xlabel("Nuclei")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
save_figure(fig, "Figure_S2_GSE157827_annotation_robustness", FIG_SUPP)
plt.show()

## Supplementary Figure S3 — Hallmark heterogeneity

In [ ]:
HALLMARK = ROOT / "GSE160936_independent_validation" / "SEAAD_vs_GSE160936_HARMONIZED_Hallmark_consensus_and_discordance_summary.csv"
require(HALLMARK)
hall = pd.read_csv(HALLMARK)

nes_cols = ["MTG_NES", "DLPFC_NES", "EC_NES", "SSC_NES"]
if not set(nes_cols).issubset(hall.columns):
    raise RuntimeError(f"Hallmark table lacks NES columns: {nes_cols}")

robust = hall.loc[
    hall.get("EC_FDR05", False).astype(bool) &
    hall.get("SSC_FDR05", False).astype(bool)
].copy()

if len(robust) == 0:
    robust = hall.reindex(hall[["EC_FDR", "SSC_FDR"]].max(axis=1).nsmallest(8).index)

robust = robust.head(8)

In [ ]:
mat = robust.set_index("pathway")[nes_cols]

fig, ax = plt.subplots(figsize=(7, 4.5))
im = ax.imshow(mat.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-3, vmax=3)
ax.set_xticks(range(4), ["MTG", "DLPFC", "EC", "SSC"])
ax.set_yticks(range(len(mat)), mat.index)
fig.colorbar(im, ax=ax, label="NES")
fig.tight_layout()
save_figure(fig, "Figure_S3_Hallmark_cross_region_heterogeneity", FIG_SUPP)
plt.show()

## Supplementary Figure S4 — GSE188545 QC and astrocyte-identification panels

Panels are saved separately for final manuscript layout. This avoids the raster re-cropping iterations that were present in the working notebook.

In [ ]:
G188 = EV / "GSE188545"
QC188 = G188 / "GSE188545_QC_scrublet_sample_summary_v1.csv"
ASTRO188 = G188 / "GSE188545_astrocyte_counts_by_donor_v1.csv"

if QC188.exists():
    q188 = pd.read_csv(QC188)
else:
    q188 = pd.DataFrame({
        "donor_id": ["AD02","AD12","AD30","AD04","AD16","AD17","HC14","HC19","HC35","HC03","HC07","HC37"],
        "cell_QC_pass_percent": [40.013657,95.015747,96.884109,96.661673,98.872434,86.872038,96.236756,93.728121,91.929285,95.963111,95.627733,98.634182],
        "final_retained_percent_of_input": [39.433254,90.853074,92.964117,92.938155,94.883877,84.723539,94.245524,89.770517,91.314374,91.874021,90.836085,93.881506],
    })

if ASTRO188.exists():
    a188 = pd.read_csv(ASTRO188)
else:
    a188 = pd.DataFrame({
        "donor_id": ["AD02","AD12","AD30","AD04","AD16","AD17","HC14","HC19","HC35","HC03","HC07","HC37"],
        "accepted_astrocytes": [198,486,761,216,147,185,276,1882,84,843,1329,420],
    })

assert int(a188["accepted_astrocytes"].sum()) == 6827

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(q188))
ax.plot(x, q188["cell_QC_pass_percent"], marker="o", label="Cell-QC pass")
ax.plot(x, q188["final_retained_percent_of_input"], marker="s", label="Final singlets")
ax.set_xticks(x, q188["donor_id"], rotation=50, ha="right")
ax.set_ylabel("Input nuclei retained (%)")
ax.legend(frameon=False, ncol=2)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_figure(fig, "Figure_S4A_QC_and_singlet_retention", FIG_SUPP)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(a188))
ax.bar(x, a188["accepted_astrocytes"])
ax.axhline(20, linestyle="--", linewidth=1)
ax.set_xticks(x, a188["donor_id"], rotation=50, ha="right")
ax.set_ylabel("Accepted astrocytes")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_figure(fig, "Figure_S4B_astrocytes_per_donor", FIG_SUPP)
plt.show()

In [ ]:
source_panels = {
    "C": G188 / "figures" / "GSE188545_UMAP_Leiden_v1.png",
    "D": G188 / "figures" / "GSE188545_UMAP_accepted_astrocytes_v1.png",
    "E": G188 / "figures" / "GSE188545_UMAP_diagnosis_audit_v1.png",
    "F": G188 / "figures" / "GSE188545_cluster_marker_zscores_v1.png",
}

for panel, src in source_panels.items():
    if src.exists():
        shutil.copy2(src, FIG_SUPP / f"Figure_S4{panel}_{src.name}")
    else:
        print(f"Optional frozen S4{panel} panel not found: {src}")

## Supplementary Figure S5 — GSE174367 model sensitivity

In [ ]:
PMI174 = EV / "GSE174367" / "GSE174367_DESeq2_plus_PMI_genomewide_v1.csv.gz"
BATCH174 = EV / "GSE174367" / "GSE174367_DESeq2_plus_batch_genomewide_v1.csv.gz"
require(DE174, PMI174, BATCH174)

models = []
for label, path in [("Primary", DE174), ("+ PMI", PMI174), ("+ Batch", BATCH174)]:
    r = creb5_row(path)
    models.append({
        "model": label,
        "log2FC": float(r["log2FoldChange"]),
        "SE": float(r["lfcSE"]),
        "pvalue": float(r["pvalue"]),
        "FDR": float(r["padj"]),
    })
sens174 = pd.DataFrame(models)
sens174["CI95_low"] = sens174["log2FC"] - 1.96*sens174["SE"]
sens174["CI95_high"] = sens174["log2FC"] + 1.96*sens174["SE"]

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ypos = np.arange(len(sens174))[::-1]
ax.errorbar(
    sens174["log2FC"], ypos,
    xerr=[sens174["log2FC"]-sens174["CI95_low"], sens174["CI95_high"]-sens174["log2FC"]],
    fmt="o", capsize=3
)
ax.axvline(0, linestyle="--", linewidth=1)
ax.set_yticks(ypos, sens174["model"])
ax.set_xlabel("CREB5 log2 fold change")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
save_figure(fig, "Figure_S5_GSE174367_CREB5_sensitivity", FIG_SUPP)
plt.show()

## Supplementary tables

In [ ]:
# S1: GSE157827 robustness
rob.to_csv(TAB_SUPP / "Supplementary_Table_S1_GSE157827_CREB5_robustness.csv", index=False)
rob.to_excel(TAB_SUPP / "Supplementary_Table_S1_GSE157827_CREB5_robustness.xlsx", index=False)

# S2: six locked secondary genes
SECONDARY = G157 / "GSE157827_STEP13B_locked_target_results_v1.csv"
require(SECONDARY)
secondary = pd.read_csv(SECONDARY)
secondary = secondary.loc[~secondary["gene"].astype(str).eq("CREB5")].copy()
secondary.to_csv(TAB_SUPP / "Supplementary_Table_S2_GSE157827_secondary_genes.csv", index=False)
secondary.to_excel(TAB_SUPP / "Supplementary_Table_S2_GSE157827_secondary_genes.xlsx", index=False)

# S3: external accession results + synthesis
with pd.ExcelWriter(TAB_SUPP / "Supplementary_Table_S3_External_Validation.xlsx") as xls:
    external.to_excel(xls, sheet_name="Accessions", index=False)
    synthesis.to_excel(xls, sheet_name="Synthesis", index=False)

## Final audit

In [ ]:
checks = {
    "8 primary effects": len(effects) == 8,
    "all primary effects positive": bool(effects["positive"].all()),
    "5 nominally significant": int(effects["nominal_p_lt_0_05"].sum()) == 5,
    "2 genome-wide FDR significant": int(effects["FDR_lt_0_05"].sum()) == 2,
    "GSE138852 positive": float(external.loc[external["dataset"].eq("GSE138852"), "log2FC"].iloc[0]) > 0,
    "fixed pooled estimate": np.isclose(fixed, 0.677998, atol=5e-6),
    "random sensitivity": np.isclose(random, 0.670385, atol=5e-6),
}

for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

assert all(checks.values())